In [ ]:
# -*- coding: utf-8 -*-
"""
最终评测脚本：指定7个模型，全量161问题，10线程并发，强制生成最终答案
"""

import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any, Optional

from openai import OpenAI
import chromadb
from sentence_transformers import SentenceTransformer

from agent import TOOLS

# ================== 配置 ==================
EVAL_DIR = "/root/autodl-tmp/毕业论文/评测"
OUTPUT_DIR = "/root/autodl-tmp/毕业论文"
MAX_ROUNDS = 5                  # 工具调用最多5轮
MAX_TOKENS = 4096
MAX_WORKERS = 10                # 并发线程数

MODEL_PATH = "/root/autodl-tmp/Qwen3-Embedding-8B"

CHROMA_PATHS = {
    "bilingual": {"path": "/root/autodl-tmp/毕业论文/chromadb_bilingual", "collection": "bilingual_sentences"},
    "official": {"path": "/root/autodl-tmp/毕业论文/chromadb_official", "collection": "official_positions"},
    "geography": {"path": "/root/autodl-tmp/毕业论文/chromadb_geo", "collection": "geography_info"},
    "event": {"path": "/root/autodl-tmp/毕业论文/chromadb_event", "collection": "event_info"},
    "term": {"path": "/root/autodl-tmp/毕业论文/chromadb_term", "collection": "term_info"},
    "kg": {"path": "/root/autodl-tmp/毕业论文/chromadb_kg", "collection": "kg_entities"}
}

# ================== 7个模型配置 ==================
MODEL_CONFIGS = [
    {
        "name": "kimi-k2.6",
        "model_id": "kimi-k2.6",
        "api_key": "YOUR_API_KEY",
        "base_url": "https://api.moonshot.cn/v1",
        "extra_body": {"enable_thinking": True},
        "reasoning_effort": None,
        "use_reasoning_effort": False,
    },
    {
        "name": "doubao-seed-2-0-pro-260215",
        "model_id": "doubao-seed-2-0-pro-260215",
        "api_key": "YOUR_API_KEY",
        "base_url": "https://ark.cn-beijing.volces.com/api/v3",
        "extra_body": {"thinking": {"type": "enabled"}},
        "reasoning_effort": None,
        "use_reasoning_effort": False,
    },
    {
        "name": "mimo-v2.5-pro",
        "model_id": "mimo-v2.5-pro",
        "api_key": "YOUR_API_KEY",
        "base_url": "https://api.xiaomimimo.com/v1",
        "extra_body": {},                         # MiMo 不支持 thinking 参数
        "reasoning_effort": None,
        "use_reasoning_effort": False,
    },
    {
        "name": "deepseek-v4-pro",
        "model_id": "deepseek-v4-pro",
        "api_key": "YOUR_API_KEY",
        "base_url": "https://api.deepseek.com",
        "extra_body": {"thinking": {"type": "enabled"}},
        "reasoning_effort": "high",
        "use_reasoning_effort": True,
    },
    {
        "name": "deepseek-v4-flash",
        "model_id": "deepseek-v4-flash",
        "api_key": "YOUR_API_KEY",
        "base_url": "https://api.deepseek.com",
        "extra_body": {"thinking": {"type": "enabled"}},
        "reasoning_effort": "high",
        "use_reasoning_effort": True,
    },
    {
        "name": "Qwen3.6-Plus",
        "model_id": "qwen3.6-plus",
        "api_key": "YOUR_API_KEY",
        "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1",
        "extra_body": {"enable_thinking": True},
        "reasoning_effort": None,
        "use_reasoning_effort": False,
    },
    {
        "name": "GLM-5.1",
        "model_id": "glm-5.1",
        "api_key": "YOUR_API_KEY",          # 使用阿里云百炼 API Key
        "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1",
        "extra_body": {"enable_thinking": True},
        "reasoning_effort": None,
        "use_reasoning_effort": False,
    },
]

# ================== 全局共享资源 ==================
shared_embed_model: Optional[SentenceTransformer] = None
shared_chroma_clients: Dict[str, Any] = {}

def load_shared_resources():
    global shared_embed_model, shared_chroma_clients
    print("正在加载嵌入模型...")
    shared_embed_model = SentenceTransformer(
        MODEL_PATH,
        tokenizer_kwargs={"padding_side": "left"},
        model_kwargs={"device_map": "auto"}
    )
    print("嵌入模型加载完成。")
    print("正在连接 ChromaDB...")
    for key, cfg in CHROMA_PATHS.items():
        try:
            client = chromadb.PersistentClient(path=cfg["path"])
            coll = client.get_collection(cfg["collection"])
            shared_chroma_clients[key] = (client, coll)
            print(f"  ✓ {cfg['collection']}")
        except Exception as e:
            print(f"  ✗ {cfg['collection']} 失败: {e}")
            shared_chroma_clients[key] = None
    print("ChromaDB 连接完成。")

def execute_tool(tool_name: str, arguments: dict) -> str:
    try:
        if tool_name == "search_bilingual":
            from 双语数据库_检索 import retrieve
            coll = shared_chroma_clients.get("bilingual")
            if coll is None: return "错误：未连接双语数据库。"
            _, collection = coll
            return retrieve(shared_embed_model, collection, arguments["query"])
        elif tool_name == "search_person":
            from 人物数据库_检索 import retrieve
            return retrieve(arguments["query"])
        elif tool_name == "search_poetry":
            from 诗文数据库_检索 import retrieve
            author = arguments.get("author", "")
            title = arguments.get("title", "")
            if not author and not title: return "错误：至少需要提供作者或标题。"
            return retrieve(author, title)
        elif tool_name == "search_official_positions":
            from 官职数据库_检索 import retrieve
            coll = shared_chroma_clients.get("official")
            if coll is None: return "错误：未连接官职数据库。"
            _, collection = coll
            return retrieve(shared_embed_model, collection, arguments["query"])
        elif tool_name == "search_geography":
            from 地理数据库_检索 import retrieve
            coll = shared_chroma_clients.get("geography")
            if coll is None: return "错误：未连接地理数据库。"
            _, collection = coll
            return retrieve(shared_embed_model, collection, arguments["query"])
        elif tool_name == "search_historical_events":
            from 典故事件数据库_检索 import retrieve
            coll = shared_chroma_clients.get("event")
            if coll is None: return "错误：未连接典故事件数据库。"
            _, collection = coll
            return retrieve(shared_embed_model, collection, arguments["query"])
        elif tool_name == "search_terminology":
            from 术语句法数据库_检索 import retrieve
            coll = shared_chroma_clients.get("term")
            if coll is None: return "错误：未连接术语数据库。"
            _, collection = coll
            return retrieve(shared_embed_model, collection, arguments["query"])
        elif tool_name == "search_knowledge_graph":
            from 知识图谱数据_检索 import retrieve
            coll = shared_chroma_clients.get("kg")
            if coll is None: return "错误：未连接知识图谱数据库。"
            _, collection = coll
            return retrieve(shared_embed_model, collection, arguments["query"])
        else:
            return f"未知工具：{tool_name}"
    except Exception as e:
        return f"工具执行错误：{str(e)}"

def test_single_question(config: dict, question_item: dict, question_index: int) -> dict:
    client = OpenAI(api_key=config["api_key"], base_url=config["base_url"])
    model_id = config["model_id"]
    extra_body = config["extra_body"]
    reasoning_effort = config.get("reasoning_effort")
    use_reasoning_effort = config.get("use_reasoning_effort", False)

    system_prompt = (
        "你是一个精通《陈书》及相关南朝历史的高级研究助手，背后接入了多源异构古籍数据库。\n"
        "你的知识库覆盖《陈书》的现代文对照、人物传记、诗文作品、官职制度、地理沿革、\n"
        "典故战争、术语解释以及知识图谱（人物、官职、地名间的关系网络）。\n\n"
        "## 核心原则（ReAct 范式）\n"
        "1. **所有回答必须基于工具返回的真实数据**，严禁凭空编造史实。\n"
        "2. **若问题明确涉及《陈书》的人物、事件、地理、制度等，你必须至少调用一次工具进行检索**，\n"
        "   即使你自认为知道答案，也必须通过工具验证或补充细节。\n"
        "3. **你可以一次性并行调用多个工具**，例如同时查人物传记和知识图谱关系，以提高效率。\n"
        "4. **完成第一轮工具调用后，仔细分析返回结果。如果信息仍不充分，或需要进一步追问细节，\n"
        "   必须继续调用工具**，但**最多进行 5 轮工具调用**。请高效规划查询，找到需要的数据后就停止，无需浪费轮次。\n"
        "5. **只有当所有必要信息均已获取，且无需更多工具时，你才能生成最终答案**。\n"
        "6. 最终答案应条理清晰，尽量引用原文依据（如古文原文或现代文出处），并注明信息来源。\n\n"
        "## 工具使用要点\n"
        "- 查询人物关系、隶属、血缘时，优先使用 `search_knowledge_graph`。\n"
        "- 将现代文描述映射回《陈书》段落，请使用 `search_bilingual`，输入完整的现代文句子。\n"
        "- 官职必须输入准确名称，诗文作者必须用规范完整姓名。\n"
        "- 如果工具返回空结果或报错，可尝试更换同义词、更规范的名称或组合其他工具。\n\n"
        "## 输出规范\n"
        "- 最终答案使用流畅的中文，适当引用古文或现代文段落。"
    )

    messages = [{"role": "system", "content": system_prompt}]
    messages.append({"role": "user", "content": question_item["question"]})
    log_entries = []
    final_answer = ""

    # 工具调用循环（最多 MAX_ROUNDS 轮）
    for round_num in range(1, MAX_ROUNDS + 1):
        try:
            api_kwargs = {
                "model": model_id,
                "messages": messages,
                "tools": TOOLS,
                "tool_choice": "auto",
                "stream": False,
                "max_tokens": MAX_TOKENS,
            }
            if extra_body:
                api_kwargs["extra_body"] = extra_body
            if use_reasoning_effort and reasoning_effort:
                api_kwargs["reasoning_effort"] = reasoning_effort

            response = client.chat.completions.create(**api_kwargs)
            msg = response.choices[0].message
            finish = response.choices[0].finish_reason

            thinking = getattr(msg, "reasoning_content", "") or ""
            log_entries.append({"round": round_num, "type": "thinking", "content": thinking})

            if finish == "tool_calls":
                messages.append(msg)
                calls = []
                for tc in msg.tool_calls:
                    name = tc.function.name
                    args = json.loads(tc.function.arguments) if tc.function.arguments else {}
                    result = execute_tool(name, args)
                    calls.append({"tool_name": name, "arguments": args, "result": result})
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                log_entries.append({"round": round_num, "type": "tool_calls", "calls": calls})
                continue

            elif finish == "stop":
                final_answer = msg.content or ""
                log_entries.append({"round": round_num, "type": "final_answer", "content": final_answer})
                messages.append(msg)
                return {
                    "index": question_index,
                    "source_file": question_item["source_file"],
                    "category": question_item["category"],
                    "question": question_item["question"],
                    "logs": log_entries,
                    "final_answer": final_answer,
                    "status": "success"
                }
            else:
                # 其他未知 finish_reason，记录错误并尝试跳出循环以触发强制生成
                log_entries.append({"round": round_num, "type": "error", "content": f"未知 finish_reason: {finish}"})
                break
        except Exception as e:
            log_entries.append({"round": round_num, "type": "error", "content": f"API 异常: {str(e)}"})
            final_answer = f"评测异常: {str(e)}"
            break

    # ---------- 强制生成最终答案 ----------
    # 若经过上述循环后仍无 final_answer，则要求模型基于已有的对话生成答案（不带工具）
    if not final_answer:
        messages.append({"role": "user", "content": "现在，请根据以上所有信息，直接给出最终答案。"})
        try:
            resp_force = client.chat.completions.create(
                model=model_id,
                messages=messages,
                stream=False,
                max_tokens=MAX_TOKENS,
                # 不传递 tools，强制模型直接回复
            )
            forced_answer = resp_force.choices[0].message.content or ""
            log_entries.append({"round": MAX_ROUNDS + 1, "type": "final_answer_forced", "content": forced_answer})
            final_answer = forced_answer
        except Exception as e:
            final_answer = f"强制生成答案失败: {e}"
            log_entries.append({"round": MAX_ROUNDS + 1, "type": "error", "content": final_answer})

    return {
        "index": question_index,
        "source_file": question_item["source_file"],
        "category": question_item["category"],
        "question": question_item["question"],
        "logs": log_entries,
        "final_answer": final_answer,
        "status": "success"  # 强制生成答案后，视为成功
    }

def load_all_questions(eval_dir: str):
    questions = []
    json_files = sorted([f for f in os.listdir(eval_dir) if f.endswith('.json')])
    print(f"发现 {len(json_files)} 个评测文件: {json_files}")
    for file_name in json_files:
        file_path = os.path.join(eval_dir, file_name)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                for item in data:
                    if isinstance(item, dict) and "question" in item:
                        questions.append({
                            "source_file": file_name,
                            "category": item.get("category", ""),
                            "question": item["question"],
                        })
    return questions

def load_existing_results(model_name: str):
    output_path = os.path.join(OUTPUT_DIR, f"{model_name}.json")
    if not os.path.exists(output_path):
        return {}
    try:
        with open(output_path, 'r', encoding='utf-8') as f:
            results_list = json.load(f)
            if isinstance(results_list, list):
                return {r.get("index"): r for r in results_list}
    except:
        return {}
    return {}

def test_model(config: dict, questions: List[dict]):
    model_name = config["name"]
    output_path = os.path.join(OUTPUT_DIR, f"{model_name}.json")
    existing = load_existing_results(model_name)
    total = len(questions)
    pending = [(i, q) for i, q in enumerate(questions) if i not in existing]
    completed = total - len(pending)

    print(f"\n[{model_name}] 全量评测：总问题 {total}，已缓存 {completed}，待处理 {len(pending)}（并发 {MAX_WORKERS}）")

    if not pending:
        print(f"[{model_name}] 全部已完成，跳过。")
        return

    results = [None] * total
    for idx, res in existing.items():
        results[idx] = res

    start_time = time.time()
    processed = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_map = {}
        for idx, q in pending:
            future = executor.submit(test_single_question, config, q, idx)
            future_map[future] = idx

        for future in as_completed(future_map):
            idx = future_map[future]
            try:
                result = future.result()
                results[idx] = result
                processed += 1

                # 实时保存
                current_results = [r for r in results if r is not None]
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(current_results, f, ensure_ascii=False, indent=2)

                elapsed = time.time() - start_time
                remaining = len(pending) - processed
                eta = (elapsed / processed) * remaining if processed > 0 else 0
                print(f"  [{model_name}] {processed}/{len(pending)} 完成 (总进度 {completed+processed}/{total})，预计剩余 {eta/60:.1f} 分钟")
            except Exception as e:
                print(f"  [{model_name}] 问题 {idx+1} 评测失败: {e}")
                results[idx] = {
                    "index": idx,
                    "source_file": questions[idx]["source_file"],
                    "category": questions[idx]["category"],
                    "question": questions[idx]["question"],
                    "logs": [],
                    "final_answer": f"线程异常: {str(e)}",
                    "status": "error"
                }

    final_results = [r for r in results if r is not None]
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(final_results, f, ensure_ascii=False, indent=2)
    print(f"[{model_name}] 评测完成，结果已保存至 {output_path}")

def main():
    load_shared_resources()
    all_questions = load_all_questions(EVAL_DIR)
    print(f"总共加载 {len(all_questions)} 个问题。")
    if len(all_questions) == 0:
        print("无有效问题，退出。")
        sys.exit(1)

    for config in MODEL_CONFIGS:
        print(f"\n{'='*60}")
        print(f"模型: {config['name']}")
        print(f"{'='*60}")
        test_model(config, all_questions)

    print("\n全部模型评测完成。")

if __name__ == "__main__":
    main()


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


正在加载嵌入模型...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

嵌入模型加载完成。
正在连接 ChromaDB...
  ✓ bilingual_sentences
  ✓ official_positions
  ✓ geography_info
  ✓ event_info
  ✓ term_info
  ✓ kg_entities
ChromaDB 连接完成。
发现 6 个评测文件: ['信息处理型.json', '多源聚合型.json', '比较分析型.json', '简单事实型.json', '综合论述型.json', '逻辑推理型.json']
总共加载 160 个问题。

模型: kimi-k2.6

[kimi-k2.6] 全量评测：总问题 160，已缓存 0，待处理 160（并发 10）


Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.556 seconds.
Prefix dict has been built successfully.


  [kimi-k2.6] 1/160 完成 (总进度 1/160)，预计剩余 49.2 分钟
  [kimi-k2.6] 2/160 完成 (总进度 2/160)，预计剩余 25.6 分钟
  [kimi-k2.6] 3/160 完成 (总进度 3/160)，预计剩余 28.9 分钟
  [kimi-k2.6] 4/160 完成 (总进度 4/160)，预计剩余 22.3 分钟
  [kimi-k2.6] 5/160 完成 (总进度 5/160)，预计剩余 18.2 分钟
  [kimi-k2.6] 6/160 完成 (总进度 6/160)，预计剩余 18.2 分钟
  [kimi-k2.6] 7/160 完成 (总进度 7/160)，预计剩余 17.1 分钟
  [kimi-k2.6] 8/160 完成 (总进度 8/160)，预计剩余 15.0 分钟
  [kimi-k2.6] 9/160 完成 (总进度 9/160)，预计剩余 14.6 分钟
  [kimi-k2.6] 10/160 完成 (总进度 10/160)，预计剩余 18.1 分钟
  [kimi-k2.6] 11/160 完成 (总进度 11/160)，预计剩余 22.1 分钟
